# 09. Statistical Evaluation: TRTR vs TSTR Benchmark

This notebook provides rigorous statistical comparison between:
- **Model A (TRTR)**: Train on Real data, Test on Real data (Baseline)
- **Model B (TSTR)**: Train on Synthetic data, Test on Real data (CVAE-augmented)

## Statistical Design

- **Patient-Level Stratification**: Bootstrap samples patients, not spectra
- **Honest Predictions**: Each patient predicted ONLY by models from its validation fold
- **No Data Leakage**: Fold i models never predict on fold i's patients (they trained on them)
- **Cached Predictions**: Predictions computed once, reused across bootstrap iterations
- **Resampling**: Stratified bootstrap at patient level (cluster bootstrap)
- **Test**: Wilcoxon signed-rank test on bootstrap differences
- **Multiple testing**: Bonferroni correction across 4 models
- **Metrics**: Accuracy, F1-macro, Cohen's κ, McNemar test

## Section 1: Imports & Setup

In [1]:
import os
import sys
import json
import logging
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import tensorflow as tf
from datetime import datetime
from pathlib import Path
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    cohen_kappa_score
)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.utils.config_manager import ConfigManager
from src.notebooks.experiment_utils import setup_experiment
from src.data.loader import GliomaDataLoader

print("✓ Imports successful")

c:\Users\Andrei\anaconda3\envs\glioma\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


✓ Imports successful


In [3]:
# Load configuration
config_name = 'exp_105'
config, output_dir = setup_experiment(config_name)

CFG = {
    "data_path": config['paths']['processed_file'],
    "split_file": config['paths']['splits_file'],
    "model_a_dir": config['paths']['models_baseline_dir'],  # TRTR
    "model_b_dir": config['paths']['models_tetr_dir'],      # TSTR
    "output_dir": config['paths']['statistical_dir'],
    "n_bootstrap": config['statistics']['bootstrap_samples'],
    "alpha": config['statistics']['alpha'],
    "random_seed": config['statistics']['random_seed'],
    "models": {
        "Apollo": {"prefix": "apollo", "ext": ".joblib", "type": "sklearn"},
        "RF_Standard": {"prefix": "rf", "ext": ".joblib", "type": "sklearn"},
        "XGBoost": {"prefix": "xgb", "ext": ".json", "type": "xgboost"},
        "1D_CNN": {"prefix": "cnn", "ext": ".keras", "type": "keras"},
    },
    "primary_metric": config['statistics']['primary_metric'],
    "n_folds": config['cv_strategy']['n_folds']
}
config["cv_strategy"]["patient_cap"] = None
np.random.seed(CFG["random_seed"])
os.makedirs(CFG["output_dir"], exist_ok=True)

print(f"✓ Configuration loaded: {config_name}")
print(f"  Bootstrap samples: {CFG['n_bootstrap']}")
print(f"  α = {CFG['alpha']}")
print(f"  Number of folds: {CFG['n_folds']}")


EXPERIMENT SETUP

Working directory: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data

 Loaded configuration: exp_105
   Description: TSAR Regimen: Fingerprint crop (700-1800), patient cap (500), and dynamic Raman augmentation

 Output directory: C:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\results\results
  Experiment already registered (re-running)

--------------------------------------------------------------------------------
Configuration Summary:
--------------------------------------------------------------------------------
  Preprocessing:
    - AirPLS lambda: 100
    - Signal length: 1738
    - Crop region: [700, 1800]

  CV Strategy:
    - Split source: data/splits/strict_splits.json
    - Use group-aware CV: True
    - Hold out test set: True

  Models:
    - RF n_estimators: 100
    - SVM C: 1.0
    - CVAE epochs: 70


✓ Configuration loaded: exp_105
  Bootstrap samples: 1000
  α = 0.05
  Number of folds: 5


## Section 2: Load Data & Build Patient-to-Fold Mapping

In [4]:
# Load data
print("Loading data...")
loader = GliomaDataLoader.from_config(
    h5_data_path=CFG['data_path'],
    config=config,
    memory_mapped=False
)

# ---------------------------------------------------------
# SECTION 2: Extract Pure Iterator Data (Perfectly Scaled)
# ---------------------------------------------------------
with open(CFG['split_file'], 'r') as f:
    splits_data = json.load(f)
splits = splits_data.get('cv_folds', splits_data)

print("\nExtracting preprocessed validation data directly from iterator...")
loader_iter = iter(loader)
all_groups = np.array(loader.groups)

X_val_list = []
y_val_list = []
patient_ids_list = []

num_classes = config['models']['cnn']['num_classes']


for fold_idx in range(CFG['n_folds']):
    fold_data = next(loader_iter)
    fold_name = fold_data['fold_name']
    
    X_val = fold_data['val'][0].astype(np.float32)
    y_val = fold_data['val'][1].copy()
    
    # Replicate Notebook 02 label mapping to ensure matching classes
    if np.min(y_val) == 1:
        y_val = y_val - 1
    
    # Apply binary conversion if needed (for exp_105)
    if num_classes == 2:
        y_val = np.where(y_val < 3, 0, 1)

    # Get patient IDs using the string lookup
    raw_string_ids = splits[fold_name]['test']
    val_indices_sorted = np.where(np.isin(all_groups, raw_string_ids))[0]
    fold_sample_ids = all_groups[val_indices_sorted]
    fold_patient_ids = np.array([s.split('_')[0] + '_' + s.split('_')[1] for s in fold_sample_ids])
    
    X_val_list.append(X_val)
    y_val_list.append(y_val)
    patient_ids_list.append(fold_patient_ids)
    print(f"  ✓ {fold_name}: {len(y_val)} spectra mapped to {len(np.unique(fold_patient_ids))} patients")

# Combine into unified master arrays for the bootstrap
y_all = np.concatenate(y_val_list)
patient_ids_all = np.concatenate(patient_ids_list)
print(f"\n✓ Unified Test Set: {len(y_all)} spectra ready for evaluation.")
if num_classes == 2:
    print(f"  ✓ Binary classification task detected (Classes: {np.unique(y_all)})")

Loading data...
Loading CV splits from: data/splits/strict_splits.json
Loading data from results\exp_102\data\glioma_dataset.h5...
[*] 'tumor_only=True': Filtering out healthy tissue...
    Filtered to 226284 Pure Tumor Spectra.
[OK] Spectral Crop: [700, 1800] cm-1 -> indices [337, 908]
  Wavenumber mapping: 699.91 -> 1799.14 cm-1
  Renormalizing to cropped region...
    After renormalization: min=0.000000, max=1.000000 (fingerprint region)
  Spectra shape after crop: (226284, 571)

Extracting preprocessed validation data directly from iterator...

 Using CV splits from: c:\Users\Andrei\incdsb\New folder\glioma-synthetic-data\data\splits\strict_splits.json
    Found 'cv_folds' section with 5 folds
   ! Separate held-out test set exists: 6 samples
   !  These samples will NOT be used in CV training/validation
   Fold 0: 160084 train samples, 45936 val samples
  ✓ fold_0: 45936 spectra mapped to 10 patients
   Fold 1: 163393 train samples, 42627 val samples
  ✓ fold_1: 42627 spectra mapp

## Section 3: Helper Functions

In [5]:
def find_model_path(base_dir: str, prefix: str, ext: str, fold_idx: int) -> str:
    """Find model file for a specific fold."""
    candidates = [
        f"{base_dir}/{prefix}_fold_{fold_idx}{ext}",
        f"{base_dir}/{prefix}_fold{fold_idx}{ext}",
        f"{base_dir}/{prefix}_{fold_idx}{ext}",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Model '{prefix}' fold {fold_idx} not found")


def load_and_predict(path: str, model_type: str, X: np.ndarray) -> np.ndarray:
    """Load model and return predictions (class labels)."""
    if model_type == "sklearn":
        model = joblib.load(path)
        return model.predict(X).astype(int)
    
    elif model_type == "xgboost":
        model = xgb.XGBClassifier()
        model.load_model(path)
        return model.predict(X).astype(int)
    
    elif model_type == "keras":
        X_3d = X.reshape(-1, X.shape[1], 1)
        model = tf.keras.models.load_model(path, compile=False)
        preds = np.argmax(model.predict(X_3d, verbose=0), axis=1)
        tf.keras.backend.clear_session()
        return preds.astype(int)
    else:
        raise ValueError(f"Unknown model type: '{model_type}'")


def get_fold_specific_predictions(preds_by_fold, patient_ids, patient_to_fold):
    """
    Extract fold-specific predictions (NO DATA LEAKAGE).
    
    For each patient, use ONLY the predictions from the fold where that patient
    was in the validation set (i.e., the fold whose model was trained WITHOUT that patient).
    
    Args:
        preds_by_fold: (n_folds, n_samples) array of predictions from each fold's model
        patient_ids: (n_samples,) array of patient IDs
        patient_to_fold: dict mapping patient_id -> fold_idx (patient's validation fold)
    
    Returns:
        preds: (n_samples,) array where preds[i] comes from the model trained on
               patient_ids[i]'s validation fold
    """
    preds = np.zeros(len(patient_ids), dtype=int)
    
    for i, patient_id in enumerate(patient_ids):
        fold_j = patient_to_fold[patient_id]
        preds[i] = preds_by_fold[fold_j][i]
    
    return preds

def stratified_patient_bootstrap(preds_A, preds_B, y_true, patient_ids, n_boot=1000, seed=42):
    """
    FAST Patient-level stratified bootstrap.
    """
    rng = np.random.default_rng(seed)
    unique_classes = np.unique(y_true)
    
    # 1. PRE-COMPUTE PATIENT INDICES (The speed secret)
    # This dictionary maps 'Patient_ID' -> array of their exact row indices
    patient_to_indices = {}
    for idx, pid in enumerate(patient_ids):
        if pid not in patient_to_indices:
            patient_to_indices[pid] = []
        patient_to_indices[pid].append(idx)
        
    for pid in patient_to_indices:
        patient_to_indices[pid] = np.array(patient_to_indices[pid])
        
    # 2. Map class -> unique patients containing that class
    class_to_patients = {}
    for cls in unique_classes:
        class_to_patients[cls] = np.unique(patient_ids[y_true == cls])
        
    acc_A_boot, acc_B_boot, boot_diffs = [], [], []

    # 3. FAST BOOTSTRAP LOOP
    print("Running optimized bootstrap loop...")
    for i in range(n_boot):
        boot_acc_A = []
        boot_acc_B = []
        
        for cls in unique_classes:
            patients_in_cls = class_to_patients[cls]
            sampled_patients = rng.choice(patients_in_cls, size=len(patients_in_cls), replace=True)
            
            for patient in sampled_patients:
                p_idx = patient_to_indices[patient]
                cls_idx = p_idx[y_true[p_idx] == cls]
                
                # Calculate accuracy for THIS specific patient
                pat_acc_A = accuracy_score(y_true[cls_idx], preds_A[cls_idx])
                pat_acc_B = accuracy_score(y_true[cls_idx], preds_B[cls_idx])
                
                boot_acc_A.append(pat_acc_A)
                boot_acc_B.append(pat_acc_B)
                
        # Average the patient accuracies (Every patient gets exactly 1 vote)
        acc_A = np.mean(boot_acc_A)
        acc_B = np.mean(boot_acc_B)
        
        acc_A_boot.append(acc_A)
        acc_B_boot.append(acc_B)
        boot_diffs.append(acc_B - acc_A)
        
    return np.array(acc_A_boot), np.array(acc_B_boot), np.array(boot_diffs)

print("✓ Helper functions defined")

✓ Helper functions defined


## Section 5: Statistical Evaluation Loop

In [8]:
# ---------------------------------------------------------
# SECTION 5: Fast Statistical Evaluation (No Data Leakage)
# ---------------------------------------------------------
results_all = []
alpha_corrected = CFG['alpha'] / len(CFG['models'])
n_folds = CFG['n_folds']

print(f"\n{'='*70}")
print(f"STATISTICAL EVALUATION: TRTR vs TSTR (Patient-Level Bootstrap)")
print(f"{'='*70}")
print(f"Bonferroni-corrected α = {alpha_corrected:.4f}\n")

for model_name, m_cfg in CFG['models'].items():
    print(f"\n{'-'*70}")
    print(f"MODEL: {model_name}")
    print(f"{'-'*70}")
    
    preds_A_list = []
    preds_B_list = []
    
    # 1. Generate strictly unseen predictions fold-by-fold
    for fold_idx in range(n_folds):
        X_val = X_val_list[fold_idx]
        
        try:
            # TRTR (Model A)
            path_A = find_model_path(CFG['model_a_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_A_list.append(load_and_predict(path_A, m_cfg['type'], X_val))
            
            # TSTR (Model B)
            path_B = find_model_path(CFG['model_b_dir'], m_cfg['prefix'], m_cfg['ext'], fold_idx)
            preds_B_list.append(load_and_predict(path_B, m_cfg['type'], X_val))
        except FileNotFoundError as e:
            print(f"  ✗ Missing model for {model_name} fold {fold_idx}")
            continue

    if len(preds_A_list) < n_folds or len(preds_B_list) < n_folds:
        print(f"⚠️  {model_name}: Incomplete predictions. Skipping.")
        continue

    # 2. Concatenate into full unseen prediction arrays
    # 2. Concatenate into full unseen prediction arrays
    preds_A = np.concatenate(preds_A_list)
    preds_B = np.concatenate(preds_B_list)
    
    # --- SALVAGE SCRIPT: Map 6-class models down to Binary ---
    # Check if TRTR (Model A) needs mapping
    # if np.max(preds_A) > 1:
    #     print("  ! Mapping TRTR predictions from 6-class to binary...")
    #     preds_A = np.where(preds_A < 3, 0, 1)
        
    # # Check if TSTR/TETR (Model B) needs mapping
    # if np.max(preds_B) > 1:
    #     print("  ! Mapping TSTR/TETR predictions from 6-class to binary...")
    #     preds_B = np.where(preds_B < 3, 0, 1)
    # ---------------------------------------------------------
    
    # 3. Compute Baseline Metrics
    acc_A = accuracy_score(y_all, preds_A)
    acc_B = accuracy_score(y_all, preds_B)
    f1_A = f1_score(y_all, preds_A, average="macro", zero_division=0)
    f1_B = f1_score(y_all, preds_B, average="macro", zero_division=0)
    kappa_A = cohen_kappa_score(y_all, preds_A)
    kappa_B = cohen_kappa_score(y_all, preds_B)
    
    print(f"Aggregate Metrics (True Unseen):")
    print(f"  TRTR: Acc={acc_A:.4f}  F1={f1_A:.4f}  κ={kappa_A:.4f}")
    print(f"  TSTR: Acc={acc_B:.4f}  F1={f1_B:.4f}  κ={kappa_B:.4f}")
    print(f"  Δ: {acc_B - acc_A:+.4f}")
    
    # 4. Patient-level stratified bootstrap
    print(f"\nRunning patient-level bootstrap ({CFG['n_bootstrap']} iterations)...")
    boot_A, boot_B, boot_diffs = stratified_patient_bootstrap(
        preds_A, preds_B, y_all, patient_ids_all,
        n_boot=CFG['n_bootstrap'], seed=CFG['random_seed']
    )
    
    # 5. Bootstrap Statistics & Wilcoxon
    mean_diff = np.mean(boot_diffs)
    median_diff = np.median(boot_diffs)
    
    # Calculate CIs for Model A (TRTR)
    ci_A_lo = np.quantile(boot_A, 0.025)
    ci_A_hi = np.quantile(boot_A, 0.975)
    
    # Calculate CIs for Model B (TSTR)
    ci_B_lo = np.quantile(boot_B, 0.025)
    ci_B_hi = np.quantile(boot_B, 0.975)
    
    # Calculate CIs for the Difference
    ci_diff_lo = np.quantile(boot_diffs, 0.025)
    ci_diff_hi = np.quantile(boot_diffs, 0.975)

    n_pos = np.sum(boot_diffs > 0)
    n_nonzero = np.sum(boot_diffs != 0)
    
    if n_nonzero > 0:
        result = binomtest(n_pos, n_nonzero, 0.5, alternative='greater')
        p_wilcoxon = result.pvalue
        z_score = (n_pos - n_nonzero/2) / np.sqrt(n_nonzero/4)
        effect_r = abs(z_score) / np.sqrt(len(y_all))
    else:
        p_wilcoxon = 1.0; effect_r = 0.0
        
    sig_corrected = p_wilcoxon < alpha_corrected
    
    # 6. McNemar Test
    correct_A = (preds_A == y_all)
    correct_B = (preds_B == y_all)
    n_11 = np.sum(correct_A & correct_B)
    n_10 = np.sum(correct_A & ~correct_B)
    n_01 = np.sum(~correct_A & correct_B)
    n_00 = np.sum(~correct_A & ~correct_B)
    
    if n_10 + n_01 > 0:
        p_mcnemar = mcnemar([[n_11, n_10], [n_01, n_00]], exact=False, correction=True).pvalue
    else:
        p_mcnemar = 1.0
        
    print(f"\nBootstrap Distribution (Δ = TSTR - TRTR):")
    print(f"  Mean:   {mean_diff:+.4f}  [95% CI: {ci_diff_lo:+.4f}, {ci_diff_hi:+.4f}]")
    print(f"Wilcoxon Test: p={p_wilcoxon:.4f} (Significant: {'✓' if sig_corrected else '✗'})")
    
    # Store results
    results_all.append({
        "model": model_name, "acc_A": float(acc_A), "acc_B": float(acc_B),
        "acc_delta": float(acc_B - acc_A), "f1_A": float(f1_A), "f1_B": float(f1_B),
        "kappa_A": float(kappa_A), "kappa_B": float(kappa_B),
        "mean_diff": float(mean_diff), "median_diff": float(median_diff),
        "ci_A_lo": float(ci_A_lo), "ci_A_hi": float(ci_A_hi),
        "ci_B_lo": float(ci_B_lo), "ci_B_hi": float(ci_B_hi), 
        "ci_diff_lo": float(ci_diff_lo), "ci_diff_hi": float(ci_diff_hi),
        "p_wilcoxon": float(p_wilcoxon), "effect_r": float(effect_r),
        "sig_corrected": bool(sig_corrected), "p_mcnemar": float(p_mcnemar),
    })


STATISTICAL EVALUATION: TRTR vs TSTR (Patient-Level Bootstrap)
Bonferroni-corrected α = 0.0125


----------------------------------------------------------------------
MODEL: Apollo
----------------------------------------------------------------------
Aggregate Metrics (True Unseen):
  TRTR: Acc=0.7057  F1=0.7057  κ=0.4158
  TSTR: Acc=0.7174  F1=0.2391  κ=0.4391
  Δ: +0.0117

Running patient-level bootstrap (1000 iterations)...
Running optimized bootstrap loop...

Bootstrap Distribution (Δ = TSTR - TRTR):
  Mean:   -0.0046  [95% CI: -0.0334, +0.0282]
Wilcoxon Test: p=1.0000 (Significant: ✗)

----------------------------------------------------------------------
MODEL: RF_Standard
----------------------------------------------------------------------
Aggregate Metrics (True Unseen):
  TRTR: Acc=0.7071  F1=0.7054  κ=0.4240
  TSTR: Acc=0.7248  F1=0.7246  κ=0.4546
  Δ: +0.0177

Running patient-level bootstrap (1000 iterations)...
Running optimized bootstrap loop...

Bootstrap Distributio

## Section 6: Save & Summarize Results

In [7]:
# Save results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = Path(CFG['output_dir']) / f"statistical_evaluation_{timestamp}.json"

with open(json_path, 'w') as f:
    json.dump(results_all, f, indent=2)

print(f"✓ Results saved: {json_path}")

# Summary table
summary_df = pd.DataFrame([
    {
        "Model": r["model"],
        "Acc(TRTR)": f"{r['acc_A']:.4f}",
        "Acc(TSTR)": f"{r['acc_B']:.4f}",
        "Δ": f"{r['acc_delta']:+.4f}",
        "p-val": f"{r['p_wilcoxon']:.4f}",
        "Sig": "✓" if r['sig_corrected'] else "",
        "Effect": f"{r['effect_r']:.3f}",
    }
    for r in results_all
])

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(summary_df.to_string(index=False))

csv_path = Path(CFG['output_dir']) / f"statistical_summary_{timestamp}.csv"
summary_df.to_csv(csv_path, index=False)
print(f"\n✓ Summary saved: {csv_path}")

✓ Results saved: results\exp_105\statistical_eval\statistical_evaluation_20260503_152707.json

SUMMARY
      Model Acc(TRTR) Acc(TSTR)       Δ  p-val Sig Effect
     Apollo    0.7057    0.5377 -0.1680 1.0000      0.070
RF_Standard    0.7071    0.7248 +0.0177 0.1342      0.003
    XGBoost    0.7343    0.5368 -0.1975 1.0000      0.070
     1D_CNN    0.6334    0.7292 +0.0957 0.0000   ✓  0.019

✓ Summary saved: results\exp_105\statistical_eval\statistical_summary_20260503_152707.csv


## Section 7: Interpretation

In [7]:
print(f"\n{'='*70}")
print("FINDINGS")
print(f"{'='*70}\n")

sig_models = [r for r in results_all if r['sig_corrected']]
improving = [r for r in results_all if r['acc_delta'] > 0]
degrading = [r for r in results_all if r['acc_delta'] < 0]

print(f"Statistically Significant Improvements (α = {alpha_corrected:.4f}):")
if sig_models:
    for r in sig_models:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")
else:
    print("  (None)")

print(f"\nImprovement Trend (not significant):")
for r in improving:
    if not r['sig_corrected']:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")

if degrading:
    print(f"\nDegradation:")
    for r in degrading:
        print(f"  • {r['model']:12s}: Δ = {r['acc_delta']:+.4f}, p = {r['p_wilcoxon']:.4f}")


FINDINGS

Statistically Significant Improvements (α = 0.0125):
  (None)

Improvement Trend (not significant):
  • 1D_CNN      : Δ = +0.0076, p = 0.1483

Degradation:
  • Apollo      : Δ = -0.1803, p = 1.0000
  • RF_Standard : Δ = -0.1133, p = 1.0000
  • XGBoost     : Δ = -0.0718, p = 1.0000
